<a href="https://colab.research.google.com/github/meerabamir240-design/flyrankai_ml_internship_task_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meerabamir240-design/flyrankai_ml_internship_task_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
!git clone https://github.com/meerabamir240-design/flyrankai_ml_internship_task_1.git
%cd flyrankai_ml_internship_task_1

Cloning into 'flyrankai_ml_internship_task_1'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 124 (delta 38), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 16.91 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/flyrankai_ml_internship_task_1/flyrankai_ml_internship_task_1/flyrankai_ml_internship_task_1


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane (Lane 2 — Refresh / Content Opportunity Scoring) is fundamentally a binary classification problem, whose output is then converted into a ranking/scoring problem.**

Underneath, the core learnable task is classification: for each content item, predict the probability that it belongs to the 'needs review' class (starter proxy: `trend_direction == "down"`). That's a yes/no label, so it's classification, not clustering (I'm not discovering unlabeled groups) and not regression (I'm not predicting a continuous number like exact future traffic).

But a raw classification probability isn't the final deliverable — a reviewer doesn't want a 0/1 flag on 30,000 pages, they want an ordered list they can work down from the top given limited weekly capacity. So the model's probability output gets combined with baseline signal scores into one `final_refresh_score`, and that combined score is what gets **ranked**. This mirrors exactly how the starter pipeline is built: `03_train_model.py` produces a classification probability, and `04_evaluate_and_export.py` turns that into a ranked, scored queue.

So: **classification model, scoring/ranking output.**

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I'd predict (starter version):** `is_declining_label = (trend_direction == "down")` — a binary label, exactly as the starter pipeline defines it.

**Where this label comes from:** this is a **proxy label**, not an observed future outcome. `trend_direction` is a bucket calculated from the *current* 90-day window (it describes what already happened, not what happens next), so a model trained on it can only learn to reproduce a rule-based bucket, not truly predict the future. I'm naming this limitation openly, as the guide requires — treating a proxy as if it were a real prediction is exactly the mistake to avoid.

**A stronger label, for later weeks:** a genuine future-outcome target, following the guide's recommended shape —

```text
features from the prior 90 days -> decline (or recovery) over the next 30 days
```

That version uses `fact_content_daily_performance` to build a feature window and a separate, later target window with no overlap between them, so the label reflects something that actually happened afterward rather than a bucket computed from the same window as the features. I'm not building that yet in this notebook — just naming it as the direction the label should move toward once I have daily data joined and a leakage audit done.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**My metric: Precision@50.**

I'm picking this over plain accuracy or ROC AUC because of how the output actually gets used: a reviewer works down a ranked list until their weekly capacity runs out (the guide's own example capacity is 20-50 pages), so what matters is *how many of the top-K recommendations are actually worth acting on* — not how the model performs across all 30,000 pages, most of which nobody will ever look at.

**What 'good' means here, using the starter pipeline's own already-verified numbers** (from `outputs/model_report.md`): the baseline hand-written rule scores Precision@50 = 0.240 (about 12 of the top 50 pages are real positives). A model that can't beat ~0.24 isn't earning its complexity over a simple if-statement. The random forest in the starter pipeline reaches Precision@50 = 0.740 (about 37 of 50) — that's the bar I'd defend as 'good' for this lane: **meaningfully above the baseline's 0.240, on a held-out set of clients the model never trained on** (client-holdout, not just a random split, since pages from the same client can share patterns the model could otherwise memorize).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json, os

results_path = "outputs/model_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("Model comparison (from outputs/model_results.json):")
    print(results)
else:
    print("outputs/model_results.json not found — run scripts/run_all.py first to regenerate it.")
    print("Verified figures from outputs/model_report.md:")
    print("  baseline rules      Precision@50 = 0.240")
    print("  logistic regression Precision@50 = 0.400")
    print("  decision tree       Precision@50 = 0.540")
    print("  random forest       Precision@50 = 0.740")



outputs/model_results.json not found — run scripts/run_all.py first to regenerate it.
Verified figures from outputs/model_report.md:
  baseline rules      Precision@50 = 0.240
  logistic regression Precision@50 = 0.400
  decision tree       Precision@50 = 0.540
  random forest       Precision@50 = 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item's 90-day performance snapshot** (`content_id`), after the same eligibility filter used in w01 (`impressions_90d > 0`, `content_age_days >= 90`, de-duplicated so each `content_id` appears once). Below I load the starter slice, keep only the columns that matter for this lane's features + proxy label, and sketch what the target column looks like before any model is trained on it.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

eligible = (
    df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
    .drop_duplicates(subset="content_id")
    .copy()
)

# Sketch the proxy target column exactly as defined in Section 2
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

feature_cols = [c for c in [
    "content_id", "client_id", "impressions_90d", "sessions_90d", "content_age_days",
    "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label",
] if c in eligible.columns]

print("One row =", "one content item's 90-day snapshot")
print("Total unit-of-analysis rows:", len(eligible))
print("Target column balance (is_declining_label):")
print(eligible["is_declining_label"].value_counts(normalize=True))

eligible[feature_cols].head(10)



One row = one content item's 90-day snapshot
Total unit-of-analysis rows: 30000
Target column balance (is_declining_label):
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,1,90,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,28,445,21.2,0.06,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,46.0,0.09,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,3,257,4.9,0.16,NaN,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter baseline is itself just a weighted if-statement — a fixed linear combination of four hand-picked scores (`0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opportunity + 0.05 * depth_gap`). Three things make that too rigid for this problem:

- **Fixed weights don't hold across pages.** A page with high impressions but strong position may not need the same review priority as a page with the same impressions but poor position — a linear rule with constant weights can't express that the *importance* of one signal depends on the value of another, but a tree-based model naturally captures those interactions.
- **Real cutoffs aren't clean.** The baseline's reason codes rely on hard thresholds (`days_since_last_update >= 180`, `ctr < 0.5`, etc.) — a page at 179 days and a page at 181 days get treated completely differently even though nothing meaningfully changed. A learned model can weigh a threshold as a soft gradient instead of a cliff.
- **The evidence already shows it, without speculation.** This isn't hypothetical — the starter pipeline's own client-holdout results (Section 3) show the fixed rule reaching Precision@50 = 0.240, while a random forest on the exact same signals reaches 0.740. If a simple if-statement could already capture the pattern, a more flexible model wouldn't have room to improve that much on data it was validated against fairly (held-out clients, not just held-out rows).

**What this doesn't claim:** that the model has found a causal reason pages decline, or that it will generalize perfectly to the full ~79M-row warehouse — only that, on this validated slice, letting the data set the weights and thresholds instead of hand-picking them produces a measurably better ranked queue for a reviewer to act on.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.